In [5]:
from google.colab import files
uploaded = files.upload()

import pandas as pd

df = pd.read_csv("first inten project.csv")


Saving first inten project.csv to first inten project.csv


In [6]:
df.columns = df.columns.str.strip()
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip()


In [7]:
df["date of reservation"] = pd.to_datetime(df["date of reservation"], errors='coerce')
df["reservation_day"] = df["date of reservation"].dt.day
df["reservation_month"] = df["date of reservation"].dt.month
df["reservation_year"] = df["date of reservation"].dt.year
df = df.dropna(subset=["reservation_day", "reservation_month", "reservation_year"])
df = df.drop("date of reservation", axis=1)


In [8]:
df_original = df.copy()


In [9]:
import numpy as np

numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df = df[(df[col] >= lower) & (df[col] <= upper)]


In [10]:
important_cols = ['number of adults', 'number of children', 'car parking space',
                  'repeated', 'P-C', 'P-not-C', 'reservation_year']

for col in important_cols:
    df[col] = df_original[col]


In [11]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['booking status'] = le.fit_transform(df['booking status'])


In [12]:
df['date of reservation'] = pd.to_datetime(df['date of reservation'], errors='coerce')
df['reservation_month'] = df['date of reservation'].dt.month
df['reservation_day'] = df['date of reservation'].dt.day
df.drop(columns=['date of reservation'], inplace=True)


KeyError: 'date of reservation'

In [ ]:
df['date of reservation'] = pd.to_datetime(df['date of reservation'], errors='coerce')


In [13]:
df['date of reservation'] = pd.to_datetime(df['date of reservation'], errors='coerce')

df['reservation_day'] = df['date of reservation'].dt.day
df['reservation_month'] = df['date of reservation'].dt.month
df['reservation_weekday'] = df['date of reservation'].dt.weekday   # 0 = Monday
df['reservation_week'] = df['date of reservation'].dt.isocalendar().week
df['reservation_is_weekend'] = df['reservation_weekday'].apply(lambda x: 1 if x >= 5 else 0)
df['reservation_season'] = df['reservation_month'].apply(
    lambda x: 'Winter' if x in [12,1,2] else 'Spring' if x in [3,4,5]
    else 'Summer' if x in [6,7,8] else 'Fall'
)


KeyError: 'date of reservation'

In [14]:
if 'Booking_ID' in df.columns:
    df = df.drop('Booking_ID', axis=1)


In [15]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

X = df.drop('booking status', axis=1)
X = X.select_dtypes(include=['int64', 'float64'])
X = X.replace([np.inf, -np.inf], np.nan)
X = X.dropna()

X_const = add_constant(X)

vif_data = pd.DataFrame()
vif_data["Feature"] = X_const.columns
vif_data["VIF"] = [variance_inflation_factor(X_const.values, i) for i in range(X_const.shape[1])]

vif_data.sort_values(by="VIF", ascending=False)


/usr/local/lib/python3.11/dist-packages/statsmodels/regression/linear_model.py:1782: RuntimeWarning: divide by zero encountered in scalar divide
  return 1 - self.ssr/self.centered_tss
/usr/local/lib/python3.11/dist-packages/statsmodels/regression/linear_model.py:1782: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss


,Feature,VIF
5,lead time,1.120925
12,reservation_month,1.119621
9,average price,1.085270
10,special requests,1.064004
3,number of week nights,1.034029
2,number of weekend nights,1.016093
11,reservation_day,1.004959
0,number of adults,0.000000
13,reservation_year,0.000000
1,number of children,NaN


In [16]:
import pandas as pd
import numpy as np
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

X = df.drop('booking status', axis=1)
X = X.select_dtypes(include=['int64', 'float64'])

X = X.replace([np.inf, -np.inf], np.nan)
X = X.dropna()

corr_matrix = X.corr()
high_corr = corr_matrix[(corr_matrix > 0.9) & (corr_matrix < 1.0)]

print(" Highly Correlated Features (corr > 0.9):")
print(high_corr[high_corr.notnull().any(axis=1)])

X_const = add_constant(X)
vif_data = pd.DataFrame()
vif_data["Feature"] = X_const.columns
vif_data["VIF"] = [variance_inflation_factor(X_const.values, i) for i in range(X_const.shape[1])]

print("\n Variance Inflation Factors (VIF):")
print(vif_data.sort_values(by="VIF", ascending=False))


 Highly Correlated Features (corr > 0.9):
Empty DataFrame
Columns: [number of adults, number of children, number of weekend nights, number of week nights, car parking space, lead time, repeated, P-C, P-not-C, average price, special requests, reservation_day, reservation_month, reservation_year]
Index: []

 Variance Inflation Factors (VIF):
                     Feature       VIF
5                  lead time  1.120925
12         reservation_month  1.119621
9              average price  1.085270
10          special requests  1.064004
3      number of week nights  1.034029
2   number of weekend nights  1.016093
11           reservation_day  1.004959
0           number of adults  0.000000
13          reservation_year  0.000000
1         number of children       NaN
4          car parking space       NaN
6                   repeated       NaN
7                        P-C       NaN
8                    P-not-C       NaN


/usr/local/lib/python3.11/dist-packages/statsmodels/regression/linear_model.py:1782: RuntimeWarning: divide by zero encountered in scalar divide
  return 1 - self.ssr/self.centered_tss
/usr/local/lib/python3.11/dist-packages/statsmodels/regression/linear_model.py:1782: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss


In [17]:
df_original['date of reservation'] = pd.to_datetime(df_original['date of reservation'], errors='coerce')

df_original['reservation_year'] = df_original['date of reservation'].dt.year

important_cols = ['number of adults', 'number of children', 'car parking space',
                  'repeated', 'P-C', 'P-not-C', 'reservation_year']

for col in important_cols:
    df[col] = df_original[col]


KeyError: 'date of reservation'

In [18]:
important_cols = ['number of adults', 'number of children', 'car parking space',
                  'repeated', 'P-C', 'P-not-C', 'reservation_year']

for col in important_cols:
    print(f"{col}: {df[col].nunique()} unique values")


number of adults: 1 unique values
number of children: 1 unique values
car parking space: 1 unique values
repeated: 1 unique values
P-C: 1 unique values
P-not-C: 1 unique values
reservation_year: 1 unique values


In [19]:
df.drop(
    ['number of adults', 'number of children', 'car parking space',
     'repeated', 'P-C', 'P-not-C', 'reservation_year'],
    axis=1, inplace=True
)


In [20]:
categorical_cols = df.select_dtypes(include='object').columns.tolist()
categorical_cols = [col for col in categorical_cols if col != 'booking status']


In [21]:
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)


In [22]:
print(" Final Columns After Encoding:\n")
print(df.columns.tolist())

print("\n Data Types:")
print(df.dtypes.value_counts())


 Final Columns After Encoding:

['number of weekend nights', 'number of week nights', 'lead time', 'average price', 'special requests', 'booking status', 'reservation_day', 'reservation_month', 'type of meal_Meal Plan 2', 'type of meal_Not Selected', 'room type_Room_Type 2', 'room type_Room_Type 3', 'room type_Room_Type 4', 'room type_Room_Type 5', 'room type_Room_Type 6', 'room type_Room_Type 7', 'market segment type_Complementary', 'market segment type_Corporate', 'market segment type_Offline', 'market segment type_Online']

 Data Types:
bool       12
int64       5
float64     3
Name: count, dtype: int64


In [23]:
from sklearn.preprocessing import StandardScaler

X = df.drop('booking status', axis=1)
y = df['booking status']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [24]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)


In [25]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(" Accuracy:", accuracy_score(y_test, y_pred))
print("\n Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\n Classification Report:\n", classification_report(y_test, y_pred))


 Accuracy: 0.7860900058445354

 Confusion Matrix:
 [[ 902  428]
 [ 304 1788]]

 Classification Report:
               precision    recall  f1-score   support

           0       0.75      0.68      0.71      1330
           1       0.81      0.85      0.83      2092

    accuracy                           0.79      3422
   macro avg       0.78      0.77      0.77      3422
weighted avg       0.78      0.79      0.78      3422



In [26]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

print(" Accuracy:", accuracy_score(y_test, y_pred_rf))
print("\n Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))
print("\n Classification Report:\n", classification_report(y_test, y_pred_rf))


 Accuracy: 0.8793103448275862

 Confusion Matrix:
 [[1083  247]
 [ 166 1926]]

 Classification Report:
               precision    recall  f1-score   support

           0       0.87      0.81      0.84      1330
           1       0.89      0.92      0.90      2092

    accuracy                           0.88      3422
   macro avg       0.88      0.87      0.87      3422
weighted avg       0.88      0.88      0.88      3422



In [27]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

importances = rf_model.feature_importances_
feature_names = df.drop('booking status', axis=1).columns

feat_imp = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print(feat_imp.head(10))


                        Feature  Importance
2                     lead time    0.346648
3                 average price    0.164421
4              special requests    0.125542
5               reservation_day    0.101641
6             reservation_month    0.087835
1         number of week nights    0.052792
0      number of weekend nights    0.035526
18   market segment type_Online    0.023729
17  market segment type_Offline    0.021934
8     type of meal_Not Selected    0.012861


In [28]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)


RandomForestClassifier(random_state=42)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
import seaborn as sns


In [29]:
X = df.drop('booking status', axis=1)
y = df['booking status']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)
